# WaveTrader — Comprehensive Strategy Testing

**All 5 strategies × 3 balance levels × multiple exit modes × all pairs**

| Strategy | Entry TF | Category |
|----------|----------|----------|
| `fib_scalper` | 1min | Scalper |
| `news_catalyst_ob` | 15min | Scalper |
| `opening_break_retest` | 15min | Scalper |
| `price_action_reversal` | 4h | Swing |
| `harmonic_scanner` | 1h | Swing |

**Balances:** $100, $500, $1000 @ 10% risk  
**Exit modes:** tp_sl, geometric_trail, multi_tp_trail  
**Multi-TP variants:** (2R, 50%), (3R, 70%), (2R,40%)+(4R,30%), (1.5R,30%)+(3R,40%)  
**Trail activation:** 1R, 2R, 3R  
**Trailing stop %:** 0.3, 0.4, 0.5

## 1. Setup — Clone Repo & Install

In [ ]:
import os, subprocess, sys

# Clone repo (skip if already exists)
REPO_DIR = "phase_lm"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/dectrick-mcgee/phase_lm.git {REPO_DIR}
    print("Cloned repo")
else:
    # Pull latest
    !cd {REPO_DIR} && git pull
    print("Repo already exists, pulled latest")

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Install requirements
!pip install -q pandas numpy torch --upgrade 2>/dev/null
print("Dependencies ready")

## 2. Load Data & Registry

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product

from wavetrader.strategies.registry import get_strategy_registry
from wavetrader.strategy_backtest import run_strategy_backtest
from wavetrader.backtest import BacktestEngine
from wavetrader.config import BacktestConfig
from wavetrader.types import TradeSignal
from wavetrader.strategies.indicators import compute_all_indicators, _ema
from wavetrader.indicators import calculate_rsi, calculate_atr, calculate_adx
from wavetrader.amd_features import compute_engulfing_patterns

reg = get_strategy_registry()
processed_dir = Path("processed_data/test")

ALL_TFS = ["1min", "15min", "1h", "4h", "1d"]
TF_FILE = {"1min": "1m", "15min": "15m", "1h": "1h", "4h": "4h", "1d": "1d"}

PAIRS = {
    "GBP/JPY": {"tag": "GBPJPY", "pip_size": 0.01, "pip_value": 6.50, "spread": 4.2},
    "EUR/JPY": {"tag": "EURJPY", "pip_size": 0.01, "pip_value": 6.50, "spread": 3.0},
    "GBP/USD": {"tag": "GBPUSD", "pip_size": 0.0001, "pip_value": 10.0, "spread": 1.5},
}


def load_pair(pair_tag: str, tfs: list) -> dict:
    """Load available timeframes for a pair."""
    out = {}
    for tf in tfs:
        p = processed_dir / f"{pair_tag}_{TF_FILE[tf]}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            if df.index.name in ("timestamp", "date", "datetime"):
                df = df.reset_index()
            df.columns = [c.strip().lower() for c in df.columns]
            col_map = {}
            for base in ("open", "high", "low", "close"):
                if f"{base}_mid" in df.columns and base not in df.columns:
                    col_map[f"{base}_mid"] = base
            if col_map:
                df = df.rename(columns=col_map)
            if "date" not in df.columns:
                for alias in ("datetime", "timestamp", "time"):
                    if alias in df.columns:
                        df = df.rename(columns={alias: "date"})
                        break
            out[tf] = df
    return out


print("Available test data:")
for pair_name, info in PAIRS.items():
    tag = info["tag"]
    for tf in ALL_TFS:
        p = processed_dir / f"{tag}_{TF_FILE[tf]}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            print(f"  {pair_name} {tf}: {len(df):,} bars")

print("\nRegistry strategies:", list(reg.list_all().keys()))

## 3. Define Test Grid

Each strategy gets tested across:
- **3 balances:** $100, $500, $1000
- **3 exit modes:** tp_sl, geometric_trail, multi_tp_trail
- **Multi-TP level variants**
- **Trail activation R levels**
- **All available pairs**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# TEST GRID
# ═══════════════════════════════════════════════════════════════════════

BALANCES = [100, 500, 1000]
RISK_PCT = 0.10  # 10% per trade

# Exit mode configurations
EXIT_CONFIGS = [
    # (label, exit_mode_override, multi_tp_levels, trail_activate_r, trailing_stop_pct_override)
    ("TP/SL default",        "tp_sl",           None,                       3.0, None),
    ("TP/SL trail@2R",       "tp_sl",           None,                       2.0, None),
    ("TP/SL trail@1R",       "tp_sl",           None,                       1.0, None),
    ("Geo trail 30%",        "geometric_trail", None,                       3.0, 0.30),
    ("Geo trail 40%",        "geometric_trail", None,                       3.0, 0.40),
    ("Geo trail 50%",        "geometric_trail", None,                       3.0, 0.50),
    ("Geo trail@2R 40%",     "geometric_trail", None,                       2.0, 0.40),
    ("Geo trail@1R 40%",     "geometric_trail", None,                       1.0, 0.40),
    ("MTP 3R-70%",           "multi_tp_trail",  ((3.0, 0.70),),             3.0, None),
    ("MTP 2R-50%",           "multi_tp_trail",  ((2.0, 0.50),),             2.0, None),
    ("MTP 2R-70%",           "multi_tp_trail",  ((2.0, 0.70),),             2.0, None),
    ("MTP 4R-70%",           "multi_tp_trail",  ((4.0, 0.70),),             3.0, None),
    ("MTP 2R40+4R30",        "multi_tp_trail",  ((2.0, 0.40), (4.0, 0.30)),3.0, None),
    ("MTP 1.5R30+3R40",      "multi_tp_trail", ((1.5, 0.30), (3.0, 0.40)), 2.0, None),
    ("MTP 2R30+3R30+5R20",   "multi_tp_trail", ((2.0, 0.30), (3.0, 0.30), (5.0, 0.20)), 2.0, None),
]

# Strategies and their required TFs / supported pairs
STRATEGY_SPECS = {
    "fib_scalper":            {"tfs": ["1min", "15min", "1h", "4h"], "pairs": ["GBP/JPY"]},  # only 1min data for GBPJPY
    "news_catalyst_ob":       {"tfs": ["15min", "1h", "4h", "1d"], "pairs": ["GBP/JPY", "EUR/JPY", "GBP/USD"]},
    "opening_break_retest":   {"tfs": ["15min", "1h", "4h"],       "pairs": ["GBP/JPY", "EUR/JPY", "GBP/USD"]},
    "price_action_reversal":  {"tfs": ["4h", "1d"],                "pairs": ["GBP/JPY", "EUR/JPY", "GBP/USD"]},
    "harmonic_scanner":       {"tfs": ["1h", "4h", "1d"],          "pairs": ["GBP/JPY", "EUR/JPY", "GBP/USD"]},
}

total_combos = sum(
    len(spec["pairs"]) * len(BALANCES) * len(EXIT_CONFIGS)
    for spec in STRATEGY_SPECS.values()
)
print(f"Total test combinations: {total_combos}")
print(f"  {len(STRATEGY_SPECS)} strategies × up to {len(PAIRS)} pairs × {len(BALANCES)} balances × {len(EXIT_CONFIGS)} exit configs")

## 4. Helper: Fib Scalper Manual Backtest

The 1-min fib_scalper needs a custom loop (skips expensive `classify_structure` on 646k bars).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# FIB SCALPER — Custom 1min backtest (fast, skips slow indicator calc)
# ═══════════════════════════════════════════════════════════════════════

MAX_1MIN_BARS = 200_000  # ~140 trading days — balance speed vs coverage


def _build_fib_indicators(df_dict: dict) -> object:
    """Build indicators for fib_scalper, skipping slow 1min structure."""
    df_1m = df_dict["1min"]
    htf_candles = {k: v for k, v in df_dict.items() if k != "1min"}
    indicators = compute_all_indicators(htf_candles, entry_tf="15min", pair="GBP/JPY", compute_amd=False)

    c = df_1m["close"].values.astype(np.float64)
    h = df_1m["high"].values.astype(np.float64)
    l = df_1m["low"].values.astype(np.float64)
    o = df_1m["open"].values.astype(np.float64)

    indicators.rsi["1min"] = calculate_rsi(c)
    indicators.atr["1min"] = calculate_atr(h, l, c)
    indicators.ema_20["1min"] = _ema(c, 20)
    indicators.ema_50["1min"] = _ema(c, 50)
    indicators.ema_200["1min"] = _ema(c, 200)
    indicators.entry_tf = "1min"
    indicators.engulfing = compute_engulfing_patterns(o, h, l, c)
    return indicators


def run_fib_backtest(df_dict: dict, indicators, params: dict,
                     cfg: BacktestConfig, exit_mode: str = "tp_sl",
                     trailing_pct_override: float = None) -> dict:
    """Run fib_scalper bar-by-bar with given config."""
    engine = BacktestEngine(cfg)
    strat = reg.instantiate('fib_scalper', params=params)
    strat.reset()
    df_1m = df_dict["1min"]

    for i in range(200, len(df_1m)):
        bar = df_1m.iloc[i]
        engine.record_bar(bar["high"], bar["low"])

        if engine.open_trade is not None:
            ts = bar.get("date", pd.Timestamp.utcnow())
            closed = engine.update_trade(bar["high"], bar["low"], bar["close"], ts)
            if closed is not None:
                continue

        if engine.open_trade is None:
            indicators.current_bar_idx = i
            setup = strat.evaluate(df_dict, indicators, i)
            if setup is not None:
                ts = bar.get("date", pd.Timestamp.utcnow())
                trail_pct = trailing_pct_override if trailing_pct_override is not None else setup.trailing_stop_pct
                trade_signal = TradeSignal(
                    signal=setup.direction,
                    confidence=setup.confidence,
                    entry_price=setup.entry_price,
                    stop_loss=setup.sl_pips,
                    take_profit=setup.tp_pips,
                    trailing_stop_pct=trail_pct,
                    timestamp=ts,
                    exit_mode=exit_mode,
                    context=getattr(setup, 'context', {}),
                    tp_levels=getattr(setup, 'tp_levels', []),
                )
                engine.open_position(trade_signal, bar["close"], ts,
                                     current_high=bar["high"], current_low=bar["low"])

    return _summarize_trades(engine.closed_trades, engine.balance, cfg.initial_balance)


def _summarize_trades(trades, final_balance, initial_balance):
    """Extract summary dict from a list of closed trades."""
    n = len(trades)
    if n == 0:
        return {"trades": 0, "wr": 0, "pf": 0, "balance": initial_balance,
                "pnl_pct": 0, "avg_w": 0, "avg_l": 0, "rr": 0, "max_w": 0, "max_l": 0}

    wins = sum(1 for t in trades if t.pnl > 0)
    total_win = sum(t.pnl for t in trades if t.pnl > 0)
    total_loss = abs(sum(t.pnl for t in trades if t.pnl <= 0))
    pf = total_win / total_loss if total_loss > 0 else 999.0
    avg_w = total_win / max(wins, 1)
    avg_l = total_loss / max(n - wins, 1)
    rr = avg_w / avg_l if avg_l > 0 else 999

    return {
        "trades": n, "wins": wins, "wr": wins / n, "pf": pf,
        "balance": final_balance, "pnl_pct": (final_balance - initial_balance) / initial_balance * 100,
        "avg_w": avg_w, "avg_l": avg_l, "rr": rr,
        "max_w": max((t.pnl for t in trades), default=0),
        "max_l": min((t.pnl for t in trades), default=0),
    }

print("Fib backtest helper ready")

## 5. Pre-compute: Load all data & indicators once

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PRE-LOAD ALL DATA (shared across all strategies)
# ═══════════════════════════════════════════════════════════════════════

DATA_CACHE = {}  # (pair_tag, tuple(tfs)) -> df_dict
FIB_INDICATORS = None  # computed once for fib_scalper
FIB_DATA = None

print("Loading data for all pairs/strategies...")
t0 = time.time()

for strat_id, spec in STRATEGY_SPECS.items():
    for pair_name in spec["pairs"]:
        tag = PAIRS[pair_name]["tag"]
        key = (tag, tuple(spec["tfs"]))
        if key not in DATA_CACHE:
            df_dict = load_pair(tag, spec["tfs"])
            DATA_CACHE[key] = df_dict
            tfs_str = ", ".join(f"{tf}:{len(df):,}" for tf, df in df_dict.items())
            print(f"  {tag} [{', '.join(spec['tfs'])}]: {tfs_str}")

# Fib scalper: trim 1min and pre-compute indicators
fib_key = ("GBPJPY", tuple(STRATEGY_SPECS["fib_scalper"]["tfs"]))
if fib_key in DATA_CACHE and "1min" in DATA_CACHE[fib_key]:
    FIB_DATA = DATA_CACHE[fib_key].copy()
    if len(FIB_DATA["1min"]) > MAX_1MIN_BARS:
        FIB_DATA["1min"] = FIB_DATA["1min"].iloc[:MAX_1MIN_BARS].copy()
        print(f"  Fib 1min trimmed to {MAX_1MIN_BARS:,} bars")
    print("  Computing fib indicators (skip 1min structure)...")
    FIB_INDICATORS = _build_fib_indicators(FIB_DATA)
    print("  Fib indicators ready")

print(f"\nAll data loaded in {time.time()-t0:.1f}s")

## 6. Run Comprehensive Grid — All Strategies

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MAIN TEST LOOP — iterates all strategies × pairs × balances × exits
# ═══════════════════════════════════════════════════════════════════════

ALL_RESULTS = []
combo_num = 0

for strat_id, spec in STRATEGY_SPECS.items():
    print(f"\n{'═' * 80}")
    print(f"STRATEGY: {strat_id}")
    print(f"{'═' * 80}")

    for pair_name in spec["pairs"]:
        pair_info = PAIRS[pair_name]
        tag = pair_info["tag"]
        data_key = (tag, tuple(spec["tfs"]))
        df_dict = DATA_CACHE.get(data_key, {})
        if not df_dict:
            print(f"  {pair_name}: NO DATA — skipping")
            continue

        for balance in BALANCES:
            for exit_label, exit_mode, mtp_levels, trail_r, trail_pct in EXIT_CONFIGS:
                combo_num += 1

                # Build config
                cfg = BacktestConfig()
                cfg.initial_balance = float(balance)
                cfg.risk_per_trade = RISK_PCT
                cfg.trail_activate_r = trail_r
                cfg.pip_size = pair_info["pip_size"]
                cfg.pip_value = pair_info["pip_value"]
                cfg.spread_pips = pair_info["spread"]
                if mtp_levels is not None:
                    cfg.multi_tp_levels = mtp_levels

                try:
                    if strat_id == "fib_scalper":
                        # Use manual bar-by-bar loop
                        if FIB_DATA is None or FIB_INDICATORS is None:
                            result = {"trades": 0, "wr": 0, "pf": 0,
                                      "balance": balance, "pnl_pct": 0}
                        else:
                            result = run_fib_backtest(
                                FIB_DATA, FIB_INDICATORS, {},
                                cfg, exit_mode=exit_mode,
                                trailing_pct_override=trail_pct,
                            )
                    else:
                        # Standard strategy backtest
                        strat = reg.instantiate(strat_id)
                        # Override exit_mode via strategy params if supported
                        if hasattr(strat, 'params'):
                            strat.params['exit_mode'] = exit_mode
                            if trail_pct is not None:
                                strat.params['trailing_stop_pct'] = trail_pct

                        results = run_strategy_backtest(
                            strat, df_dict, bt_config=cfg,
                            pair=pair_name, verbose=False,
                        )
                        result = _summarize_trades(
                            results.trades, results.final_balance, cfg.initial_balance
                        )

                except Exception as e:
                    result = {"trades": 0, "wr": 0, "pf": 0,
                              "balance": balance, "pnl_pct": 0, "error": str(e)}

                result.update({
                    "strategy": strat_id,
                    "pair": pair_name,
                    "balance_start": balance,
                    "exit_config": exit_label,
                    "exit_mode": exit_mode,
                    "trail_r": trail_r,
                })
                ALL_RESULTS.append(result)

                # Progress print (compact)
                n = result.get("trades", 0)
                if n > 0:
                    wr = result["wr"]
                    pf = result["pf"]
                    bal = result["balance"]
                    pct = result["pnl_pct"]
                    if combo_num % 20 == 0 or pct > 100:  # Print milestones + big winners
                        print(f"  [{combo_num:>4}] {strat_id:<25} {pair_name:<8} ${balance:>5} {exit_label:<22} "
                              f"{n:>3}T WR={wr:.0%} PF={pf:.2f} → ${bal:,.0f} ({pct:+.0f}%)")

print(f"\n{'═' * 80}")
print(f"COMPLETE: {combo_num} combinations tested")
print(f"{'═' * 80}")

## 7. Results Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Convert to DataFrame for analysis
# ═══════════════════════════════════════════════════════════════════════

df_results = pd.DataFrame(ALL_RESULTS)
df_results = df_results[df_results["trades"] > 0].copy()  # drop zero-trade combos

print(f"Total results with trades: {len(df_results)} / {len(ALL_RESULTS)}")
print(f"Strategies: {df_results['strategy'].unique().tolist()}")
print(f"Pairs: {df_results['pair'].unique().tolist()}")
df_results.head()

### 7a. Best Configuration per Strategy × Balance

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BEST CONFIG PER STRATEGY × BALANCE (by final PnL %)
# ═══════════════════════════════════════════════════════════════════════

print(f"{'Strategy':<25} {'Bal':>5} {'Pair':<8} {'Exit Config':<22} {'Trades':>6} {'WR':>5} {'PF':>6} {'Final $':>10} {'PnL%':>8} {'R:R':>5}")
print("─" * 110)

for strat_id in STRATEGY_SPECS.keys():
    for bal in BALANCES:
        subset = df_results[(df_results["strategy"] == strat_id) & (df_results["balance_start"] == bal)]
        if subset.empty:
            print(f"{strat_id:<25} ${bal:>4}  — no trades")
            continue
        best = subset.loc[subset["pnl_pct"].idxmax()]
        print(f"{strat_id:<25} ${bal:>4} {best['pair']:<8} {best['exit_config']:<22} "
              f"{best['trades']:>6.0f} {best['wr']:>4.0%} {best['pf']:>6.2f} "
              f"${best['balance']:>9,.0f} {best['pnl_pct']:>+7.0f}% {best.get('rr', 0):>5.2f}")

### 7b. Best Configuration per Balance Level (across ALL strategies)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BEST OVERALL per balance level
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "═" * 100)
print("BEST OVERALL BY BALANCE")
print("═" * 100)

for bal in BALANCES:
    subset = df_results[df_results["balance_start"] == bal]
    if subset.empty:
        print(f"  ${bal}: No results")
        continue

    # Top 5 by PnL%
    top5 = subset.nlargest(5, "pnl_pct")
    print(f"\n  ── ${bal} Balance ── Top 5 by PnL% ──")
    for rank, (_, row) in enumerate(top5.iterrows(), 1):
        print(f"    #{rank} {row['strategy']:<25} {row['pair']:<8} {row['exit_config']:<22} "
              f"{row['trades']:.0f}T WR={row['wr']:.0%} PF={row['pf']:.2f} "
              f"→ ${row['balance']:,.0f} ({row['pnl_pct']:+,.0f}%)")

### 7c. Exit Mode Comparison (Heatmap)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EXIT MODE COMPARISON — Pivot table: strategy × exit_config → avg PnL%
# ═══════════════════════════════════════════════════════════════════════

# Use $100 balance for comparison (most sensitive to config)
df_100 = df_results[df_results["balance_start"] == 100].copy()

if not df_100.empty:
    pivot = df_100.pivot_table(
        values="pnl_pct", index="strategy", columns="exit_config",
        aggfunc="mean"
    ).round(0)

    # Style: highlight best per row
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: #2ecc71; font-weight: bold' if v else '' for v in is_max]

    print("\nAverage PnL% by Strategy × Exit Config ($100 balance):")
    display(pivot.style.apply(highlight_max, axis=1).format("{:.0f}%"))
else:
    print("No $100 results available")

### 7d. Trailing Stop vs Multi-TP Deep Dive

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# TRAILING STOP % ANALYSIS — which trailing % works best per strategy?
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "═" * 90)
print("TRAILING STOP & MULTI-TP DEEP DIVE ($100 balance)")
print("═" * 90)

# Group: geometric_trail configs by trailing %
geo_results = df_100[df_100["exit_mode"] == "geometric_trail"]
if not geo_results.empty:
    print("\n  ── Geometric Trail: PnL% by trailing stop % ──")
    for strat_id in STRATEGY_SPECS:
        strat_geo = geo_results[geo_results["strategy"] == strat_id]
        if strat_geo.empty:
            continue
        best = strat_geo.loc[strat_geo["pnl_pct"].idxmax()]
        configs = strat_geo.groupby("exit_config")["pnl_pct"].mean().to_dict()
        cfg_str = " | ".join(f"{k}: {v:+.0f}%" for k, v in sorted(configs.items()))
        print(f"    {strat_id:<25} Best: {best['exit_config']} ({best['pnl_pct']:+.0f}%)")
        print(f"    {'':25} All: {cfg_str}")

# Group: multi_tp_trail configs
mtp_results = df_100[df_100["exit_mode"] == "multi_tp_trail"]
if not mtp_results.empty:
    print("\n  ── Multi-TP Trail: PnL% by TP config ──")
    for strat_id in STRATEGY_SPECS:
        strat_mtp = mtp_results[mtp_results["strategy"] == strat_id]
        if strat_mtp.empty:
            continue
        best = strat_mtp.loc[strat_mtp["pnl_pct"].idxmax()]
        configs = strat_mtp.groupby("exit_config")["pnl_pct"].mean().to_dict()
        cfg_str = " | ".join(f"{k}: {v:+.0f}%" for k, v in sorted(configs.items()))
        print(f"    {strat_id:<25} Best: {best['exit_config']} → ${best['balance']:,.0f} ({best['pnl_pct']:+.0f}%)")
        print(f"    {'':25} All: {cfg_str}")

# TP/SL by trail activation R
tpsl_results = df_100[df_100["exit_mode"] == "tp_sl"]
if not tpsl_results.empty:
    print("\n  ── TP/SL: PnL% by trail activation R ──")
    for strat_id in STRATEGY_SPECS:
        strat_tpsl = tpsl_results[tpsl_results["strategy"] == strat_id]
        if strat_tpsl.empty:
            continue
        best = strat_tpsl.loc[strat_tpsl["pnl_pct"].idxmax()]
        configs = strat_tpsl.groupby("exit_config")["pnl_pct"].mean().to_dict()
        cfg_str = " | ".join(f"{k}: {v:+.0f}%" for k, v in sorted(configs.items()))
        print(f"    {strat_id:<25} Best: {best['exit_config']} ({best['pnl_pct']:+.0f}%)")
        print(f"    {'':25} All: {cfg_str}")

### 7e. Pair Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PAIR COMPARISON — which pair is best per strategy?
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "═" * 90)
print("BEST PAIR PER STRATEGY ($100 balance, best exit config)")
print("═" * 90)

for strat_id in STRATEGY_SPECS:
    subset = df_100[df_100["strategy"] == strat_id]
    if subset.empty:
        print(f"  {strat_id:<25} — no results")
        continue
    pair_bests = subset.groupby("pair").apply(
        lambda g: g.loc[g["pnl_pct"].idxmax()]
    )
    overall_best = subset.loc[subset["pnl_pct"].idxmax()]
    print(f"\n  {strat_id}:")
    for _, row in pair_bests.iterrows():
        marker = " ★" if row["pair"] == overall_best["pair"] and row["exit_config"] == overall_best["exit_config"] else ""
        print(f"    {row['pair']:<8} {row['exit_config']:<22} {row['trades']:.0f}T WR={row['wr']:.0%} PF={row['pf']:.2f} "
              f"→ ${row['balance']:,.0f} ({row['pnl_pct']:+,.0f}%){marker}")

## 8. Final Summary — Max Profit Configs

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# FINAL SUMMARY — MAX PROFIT CONFIGURATION FOR EACH BALANCE TIER
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "█" * 90)
print("  MAX PROFIT CONFIGURATIONS")
print("█" * 90)

for bal in BALANCES:
    subset = df_results[df_results["balance_start"] == bal]
    if subset.empty:
        continue

    best = subset.loc[subset["pnl_pct"].idxmax()]

    print(f"\n  ┌──────────────────────────────────────────────────────┐")
    print(f"  │  ${bal:>5} BALANCE → ${best['balance']:>10,.0f}  ({best['pnl_pct']:>+,.0f}%)       │")
    print(f"  ├──────────────────────────────────────────────────────┤")
    print(f"  │  Strategy:  {best['strategy']:<40} │")
    print(f"  │  Pair:      {best['pair']:<40} │")
    print(f"  │  Exit Mode: {best['exit_config']:<40} │")
    print(f"  │  Trades:    {best['trades']:<40.0f} │")
    print(f"  │  Win Rate:  {best['wr']:<40.0%} │")
    print(f"  │  PF:        {best['pf']:<40.2f} │")
    print(f"  │  R:R:       {best.get('rr', 0):<40.2f} │")
    print(f"  └──────────────────────────────────────────────────────┘")

# Also show top 10 overall
print(f"\n\n{'═' * 90}")
print("TOP 10 CONFIGURATIONS OVERALL (by PnL%)")
print(f"{'═' * 90}")
print(f"{'#':>3} {'Strategy':<25} {'Pair':<8} {'$Start':>6} {'Exit Config':<22} {'Trades':>6} {'WR':>5} {'PF':>6} {'Final$':>10} {'PnL%':>8}")
print("─" * 110)

top10 = df_results.nlargest(10, "pnl_pct")
for rank, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"{rank:>3} {row['strategy']:<25} {row['pair']:<8} ${row['balance_start']:>5} {row['exit_config']:<22} "
          f"{row['trades']:>6.0f} {row['wr']:>4.0%} {row['pf']:>6.2f} ${row['balance']:>9,.0f} {row['pnl_pct']:>+7,.0f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SAVE RESULTS TO CSV
# ═══════════════════════════════════════════════════════════════════════

out_path = "backtest_results/comprehensive_strategy_test.csv"
df_all = pd.DataFrame(ALL_RESULTS)
df_all.to_csv(out_path, index=False)
print(f"\nResults saved to {out_path}")
print(f"Total rows: {len(df_all)} ({len(df_results)} with trades)")